In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [29]:
# ─────────────────────────────────────────────
# LOAD & PREPARE DATA
# ─────────────────────────────────────────────
df = pd.read_csv("human_vital_signs_dataset_2024_cleaned.csv")

X = df.drop(columns=["risk_category"])
y = df["risk_category"]

le_gender = LabelEncoder()
X = X.copy()
X["gender"] = le_gender.fit_transform(X["gender"])

risk_mapping = {"Low Risk": 0,  "High Risk": 1}
y_encoded = y.map(risk_mapping)
class_names = ["Low Risk",  "High Risk"]
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)


In [30]:
# ─────────────────────────────────────────────
# TEST 1: BASIC ACCURACY
# ─────────────────────────────────────────────
print("=" * 55)
print("TEST 1: BASIC ACCURACY")
print("=" * 55)
train_acc = accuracy_score(y_train, rf_model.predict(X_train))
test_acc  = accuracy_score(y_test, y_pred)

print(f"Train Accuracy : {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"Difference     : {abs(train_acc - test_acc):.4f}")

if abs(train_acc - test_acc) > 0.05:
    print("⚠️  Large gap → possible overfitting")
else:
    print("✅ Small gap → model generalizes well")

TEST 1: BASIC ACCURACY
Train Accuracy : 1.0000 (100.00%)
Test  Accuracy : 1.0000  (100.00%)
Difference     : 0.0000
✅ Small gap → model generalizes well


In [31]:
# ─────────────────────────────────────────────
# TEST 2: CLASSIFICATION REPORT
# (Precision, Recall, F1 per class)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 2: CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=class_names))



TEST 2: CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00     18981
   High Risk       1.00      1.00      1.00     21023

    accuracy                           1.00     40004
   macro avg       1.00      1.00      1.00     40004
weighted avg       1.00      1.00      1.00     40004



In [32]:
# ─────────────────────────────────────────────
# TEST 3: CONFUSION MATRIX
# ─────────────────────────────────────────────
print("=" * 55)
print("TEST 3: CONFUSION MATRIX")
print("=" * 55)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("(Rows = Actual class, Columns = Predicted class)")
print("Diagonal = correct predictions, Off-diagonal = errors\n")

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.close()
print("✅ Confusion matrix saved to confusion_matrix.png")


TEST 3: CONFUSION MATRIX
[[18981     0]
 [    0 21023]]
(Rows = Actual class, Columns = Predicted class)
Diagonal = correct predictions, Off-diagonal = errors

✅ Confusion matrix saved to confusion_matrix.png


In [33]:
# ─────────────────────────────────────────────
# TEST 4: CROSS VALIDATION
# (Most important test — checks if 99% is real)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 4: CROSS VALIDATION (5-Fold)")
print("=" * 55)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y_encoded, cv=cv, scoring="accuracy", n_jobs=-1)

print(f"CV Scores per fold : {[round(s, 4) for s in cv_scores]}")
print(f"Mean CV Accuracy   : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)")
print(f"Std Dev            : {cv_scores.std():.4f}")

if cv_scores.std() < 0.02:
    print("✅ Low std dev → model is consistent across folds")
else:
    print("⚠️  High std dev → model performance is unstable")



TEST 4: CROSS VALIDATION (5-Fold)
CV Scores per fold : [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
Mean CV Accuracy   : 1.0000 (100.00%)
Std Dev            : 0.0000
✅ Low std dev → model is consistent across folds


In [34]:
# ─────────────────────────────────────────────
# TEST 5: OVERFITTING CHECK
# Train on smaller data, test on more
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 5: OVERFITTING CHECK (Learning Curve)")
print("=" * 55)
train_sizes = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
train_accs, test_accs = [], []

for size in train_sizes:
    if size < 1.0:
        X_tr, _, y_tr, _ = train_test_split(X_train, y_train, train_size=size, random_state=42, stratify=y_train)
    else:
        X_tr, y_tr = X_train, y_train

    rf_temp = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rf_temp.fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, rf_temp.predict(X_tr)))
    test_accs.append(accuracy_score(y_test, rf_temp.predict(X_test)))
    print(f"  Train size {int(size*100):>3}% → Train acc: {train_accs[-1]:.4f} | Test acc: {test_accs[-1]:.4f}")



TEST 5: OVERFITTING CHECK (Learning Curve)
  Train size  10% → Train acc: 1.0000 | Test acc: 0.9999
  Train size  20% → Train acc: 1.0000 | Test acc: 1.0000
  Train size  40% → Train acc: 1.0000 | Test acc: 1.0000
  Train size  60% → Train acc: 1.0000 | Test acc: 1.0000
  Train size  80% → Train acc: 1.0000 | Test acc: 1.0000
  Train size 100% → Train acc: 1.0000 | Test acc: 1.0000


In [35]:
# Plot learning curve
plt.figure(figsize=(8, 5))
plt.plot([s * 100 for s in train_sizes], train_accs, "o-", label="Train Accuracy", color="blue")
plt.plot([s * 100 for s in train_sizes], test_accs,  "o-", label="Test Accuracy",  color="orange")
plt.xlabel("Training Data Used (%)")
plt.ylabel("Accuracy")
plt.title("Learning Curve", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curve.png", dpi=150)
plt.close()
print("✅ Learning curve saved to learning_curve.png")


✅ Learning curve saved to learning_curve.png


In [36]:
# PLOT 1: SINGLE TREE (Tree #0) — Full depth
# ─────────────────────────────────────────────
from sklearn import tree
single_tree = rf_model.estimators_[0]

plt.figure(figsize=(40, 20))
tree.plot_tree(
    single_tree,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,           # color nodes by majority class
    rounded=True,          # rounded boxes
    fontsize=9,
    max_depth=4,           # show only top 4 levels (full tree is too large)
    impurity=True,         # show gini impurity
    proportion=False
)
plt.title("Random Forest — Single Tree Visualization (Top 4 Levels)", 
          fontsize=18, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("rf_single_tree.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Saved: rf_single_tree.png")

✅ Saved: rf_single_tree.png
